# Notebook 26: Offline Branching Trajectory Lab

Offline trajectory analysis for the multi-agent branching idea. This notebook does **not** call the LLM API. It treats the existing Notebook `13`, Notebook `24`, and Notebook `25` live base trajectories as observed samples from the same Notebook `13`-style workup process, then asks:

- where do trajectories diverge under identical visible evidence?
- how often does a branch contain a correct answer when the base path misses?
- can label-free uncertainty, graph, Bayes, and MLP signals identify when to branch?
- which branch adjudicators are plausible enough for a future live confirmation?

This is a policy lab and feasibility notebook, not a promoted live result.


## 1. Utility Functions

In [ ]:
from pathlib import Path
import ast
import json
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:  # pragma: no cover
    plt = None
    print(f"matplotlib unavailable: {exc}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "offline_branching_trajectory_lab_49case_v1"
FIGURES_DIR = ARTIFACT_ROOT / "figures"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_ORDER = ["n13_frozen", "n24_base", "n25_r01", "n25_r02", "n25_r03"]
RUN_SPECS = {
    "n13_frozen": PROJECT_ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_49case_v1",
    "n24_base": PROJECT_ROOT / "artifacts" / "graph_algorithmic_ledger" / "live_graph_bayes_rescue_confirmation_49case_v1",
    "n25_r01": PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "notebook13_style_live_base_replicates_49case_v1" / "replicate_r01",
    "n25_r02": PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "notebook13_style_live_base_replicates_49case_v1" / "replicate_r02",
    "n25_r03": PROJECT_ROOT / "artifacts" / "trajectory_replicates" / "notebook13_style_live_base_replicates_49case_v1" / "replicate_r03",
}

GRAPH_EDGE_PATH = PROJECT_ROOT / "artifacts" / "graph_algorithmic_ledger" / "medkgi_style_offline_notebook13_49case_v1" / "global_evidence_graph_edges.csv"
BAYES_LIKELIHOOD_PATH = PROJECT_ROOT / "artifacts" / "bayesian_voi_ledger" / "bayesian_voi_offline_notebook13_49case_v1" / "root_outcome_likelihoods.csv"
BAYES_PRIOR_PATH = PROJECT_ROOT / "artifacts" / "bayesian_voi_ledger" / "bayesian_voi_offline_notebook13_49case_v1" / "diagnosis_priors.csv"

GRAPH_CLIP = 3.0
EPS = 1e-12


def read_jsonl(path):
    rows = []
    with Path(path).open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def parse_jsonish(value, default=None):
    if default is None:
        default = []
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    text = str(value)
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return default


def as_bool(value):
    if isinstance(value, bool):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


def safe_float(value, default=np.nan):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default


def normalize_status(status):
    if status is None:
        return None
    text = str(status).strip().lower()
    if text in {"present", "yes", "true", "1", "__present__"}:
        return "present"
    if text in {"absent", "no", "false", "0", "__absent__"}:
        return "absent"
    if text.startswith("yes"):
        return "present"
    if text.startswith("no"):
        return "absent"
    return text


def bayes_state(status):
    status = normalize_status(status)
    if status == "present":
        return "__PRESENT__"
    if status == "absent":
        return "__ABSENT__"
    return None


def parse_initial_ledger(visible_ledger):
    evidence = {}
    if not isinstance(visible_ledger, str):
        return evidence
    for line in visible_ledger.splitlines():
        match = re.search(r"(E_\d+)\s*:\s*.*?->\s*([^\[]+)", line)
        if not match:
            continue
        root = match.group(1)
        raw_status = match.group(2).strip()
        status = normalize_status(raw_status)
        if status in {"present", "absent"}:
            evidence[root] = status
    return evidence


def final_evidence_from_trace(trace_obj):
    trace = trace_obj.get("trace", []) if isinstance(trace_obj, dict) else []
    evidence = {}
    if trace:
        visible = trace[0].get("visible_context_before", {}).get("visible_ledger", "")
        evidence.update(parse_initial_ledger(visible))
    for turn in trace:
        payload = turn.get("reveal_payload")
        if not isinstance(payload, dict):
            continue
        root = payload.get("root_evidence_id")
        status = normalize_status(payload.get("status"))
        if root and status in {"present", "absent"}:
            evidence[root] = status
    return evidence


def evidence_signature(evidence):
    return json.dumps(sorted(evidence.items()), separators=(",", ":"))


def softmax_dict(scores):
    if not scores:
        return {}
    vals = np.array(list(scores.values()), dtype=float)
    vals = vals - np.nanmax(vals)
    exps = np.exp(vals)
    denom = exps.sum()
    if denom <= 0 or not np.isfinite(denom):
        return {k: 1.0 / len(scores) for k in scores}
    return {k: float(v / denom) for k, v in zip(scores.keys(), exps)}


def rank_of(scores, item, descending=True):
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=descending)
    for idx, (name, _) in enumerate(ordered, start=1):
        if name == item:
            return idx
    return np.nan


def top_margin(scores):
    vals = sorted(scores.values(), reverse=True)
    if not vals:
        return np.nan
    if len(vals) == 1:
        return vals[0]
    return vals[0] - vals[1]


def true_in_topk(ranked, true_pathology, k):
    ranked = parse_jsonish(ranked, default=[])
    return true_pathology in ranked[:k]


def minmax(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return arr
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
        return np.zeros_like(arr, dtype=float)
    return (arr - lo) / (hi - lo)


## 2. Load Notebook 13, 24, And 25 Trajectories

In [ ]:
missing = []
for run_id, root in RUN_SPECS.items():
    for filename in ["predictions.csv", "traces.jsonl"]:
        path = root / filename
        if not path.exists():
            missing.append(str(path))
if missing:
    raise FileNotFoundError("Missing required trajectory artifacts:\n" + "\n".join(missing))

run_prediction_frames = {}
run_trace_maps = {}
for run_id, root in RUN_SPECS.items():
    pred = pd.read_csv(root / "predictions.csv")
    traces = {obj["case_id"]: obj for obj in read_jsonl(root / "traces.jsonl")}
    run_prediction_frames[run_id] = pred
    run_trace_maps[run_id] = traces
    print(run_id, pred.shape, len(traces), root)

common_cases = set.intersection(*[set(df["case_id"]) for df in run_prediction_frames.values()])
print(f"common cases across all runs: {len(common_cases)}")
if len(common_cases) != 49:
    raise ValueError(f"Expected 49 common cases, found {len(common_cases)}")


In [ ]:
def build_case_run_outcomes():
    records = []
    for run_id in RUN_ORDER:
        pred = run_prediction_frames[run_id].copy()
        traces = run_trace_maps[run_id]
        for _, row in pred.iterrows():
            cid = row["case_id"]
            if cid not in common_cases:
                continue
            trace_obj = traces.get(cid, {})
            ranked = parse_jsonish(row.get("ranked_differential"), default=[])
            final_evidence = final_evidence_from_trace(trace_obj)
            final_mlp = trace_obj.get("final_mlp_feedback", {}) if isinstance(trace_obj, dict) else {}
            records.append({
                "run_id": run_id,
                "case_id": cid,
                "true_pathology": row.get("true_pathology"),
                "predicted_pathology": row.get("predicted_pathology"),
                "correct": as_bool(row.get("correct")),
                "top3_correct": true_in_topk(row.get("ranked_differential"), row.get("true_pathology"), 3),
                "top5_correct": true_in_topk(row.get("ranked_differential"), row.get("true_pathology"), 5),
                "ranked_differential": json.dumps(ranked),
                "num_requests": int(row.get("num_requests", trace_obj.get("num_requests", 0))),
                "visible_root_count": int(row.get("visible_root_count", len(final_evidence))),
                "stop_reason": row.get("stop_reason", trace_obj.get("stop_reason")),
                "stop_rule_fired": as_bool(row.get("stop_rule_fired", trace_obj.get("stop_rule_fired"))),
                "final_mlp_confidence": safe_float(row.get("final_mlp_confidence", final_mlp.get("confidence"))),
                "final_mlp_margin": safe_float(row.get("final_mlp_margin", final_mlp.get("margin"))),
                "final_mlp_entropy": safe_float(row.get("final_mlp_entropy", final_mlp.get("entropy"))),
                "final_mlp_stability_turns": safe_float(row.get("final_mlp_stability_turns", final_mlp.get("stability_turns"))),
                "llm_mlp_agreement": as_bool(row.get("llm_mlp_agreement", False)),
                "llm_confidence": safe_float(row.get("llm_confidence", row.get("final_confidence"))),
                "deterministic_top1": row.get("deterministic_top1"),
                "deterministic_margin": safe_float(row.get("deterministic_margin")),
                "deterministic_unresolved_mass": safe_float(row.get("deterministic_unresolved_mass")),
                "final_top_shortlist_score": safe_float(row.get("final_top_shortlist_score")),
                "evidence_signature": evidence_signature(final_evidence),
                "visible_evidence_json": json.dumps(final_evidence, sort_keys=True),
            })
    return pd.DataFrame(records)

case_run_outcomes = build_case_run_outcomes()
print(case_run_outcomes.shape)
case_run_outcomes.head()


## 3. Reconstruct Graph And Bayesian Final-State Scores

In [ ]:
if not GRAPH_EDGE_PATH.exists():
    raise FileNotFoundError(GRAPH_EDGE_PATH)
if not BAYES_LIKELIHOOD_PATH.exists():
    raise FileNotFoundError(BAYES_LIKELIHOOD_PATH)
if not BAYES_PRIOR_PATH.exists():
    raise FileNotFoundError(BAYES_PRIOR_PATH)

graph_edges = pd.read_csv(GRAPH_EDGE_PATH)
graph_pathologies = sorted(graph_edges["pathology"].dropna().unique().tolist())
graph_lookup = {
    (row.root_evidence_id, normalize_status(row.outcome_state), row.pathology): float(row.log_odds_support)
    for row in graph_edges.itertuples(index=False)
}
print(f"graph edges: {len(graph_lookup):,}; pathologies: {len(graph_pathologies)}")

bayes_likelihoods = pd.read_csv(BAYES_LIKELIHOOD_PATH)
bayes_priors_df = pd.read_csv(BAYES_PRIOR_PATH)
bayes_pathologies = bayes_priors_df["pathology"].tolist()
bayes_priors = dict(zip(bayes_priors_df["pathology"], bayes_priors_df["prior"]))
bayes_index = bayes_likelihoods.set_index(["root_evidence_id", "outcome_state"])
print(f"bayes likelihood states: {len(bayes_index):,}; pathologies: {len(bayes_pathologies)}")

ALL_PATHOLOGIES = sorted(set(graph_pathologies) | set(bayes_pathologies))


In [ ]:
def compute_graph_scores(evidence):
    scores = {p: 0.0 for p in ALL_PATHOLOGIES}
    positive = {p: 0.0 for p in ALL_PATHOLOGIES}
    contradiction = {p: 0.0 for p in ALL_PATHOLOGIES}
    for root, status in evidence.items():
        state = normalize_status(status)
        if state not in {"present", "absent"}:
            continue
        for pathology in ALL_PATHOLOGIES:
            raw = graph_lookup.get((root, state, pathology), 0.0)
            val = float(np.clip(raw, -GRAPH_CLIP, GRAPH_CLIP))
            scores[pathology] += val
            if val >= 0:
                positive[pathology] += val
            else:
                contradiction[pathology] += -val
    posterior = softmax_dict(scores)
    return scores, positive, contradiction, posterior


def compute_bayes_scores(evidence):
    scores = {p: math.log(max(float(bayes_priors.get(p, EPS)), EPS)) for p in ALL_PATHOLOGIES}
    for root, status in evidence.items():
        state = bayes_state(status)
        if state is None or (root, state) not in bayes_index.index:
            continue
        like_row = bayes_index.loc[(root, state)]
        if isinstance(like_row, pd.DataFrame):
            like_row = like_row.iloc[0]
        for pathology in ALL_PATHOLOGIES:
            col = f"p__{pathology}"
            if col not in like_row.index:
                continue
            prob = max(float(like_row[col]), EPS)
            scores[pathology] += math.log(prob)
    posterior = softmax_dict(scores)
    return scores, posterior


def enrich_with_ledger_scores(row):
    evidence = parse_jsonish(row["visible_evidence_json"], default={})
    predicted = row["predicted_pathology"]

    graph_scores, graph_pos, graph_contra, graph_post = compute_graph_scores(evidence)
    graph_order = sorted(graph_scores.items(), key=lambda kv: kv[1], reverse=True)
    graph_top1, graph_top_score = graph_order[0]
    pred_graph_score = graph_scores.get(predicted, 0.0)

    bayes_scores, bayes_post = compute_bayes_scores(evidence)
    bayes_order = sorted(bayes_scores.items(), key=lambda kv: kv[1], reverse=True)
    bayes_top1, bayes_top_score = bayes_order[0]
    pred_bayes_score = bayes_scores.get(predicted, float("-inf"))

    return pd.Series({
        "graph_top1": graph_top1,
        "graph_top_score": graph_top_score,
        "graph_margin": top_margin(graph_scores),
        "pred_graph_score": pred_graph_score,
        "pred_graph_rank": rank_of(graph_scores, predicted),
        "pred_graph_posterior": graph_post.get(predicted, 0.0),
        "pred_graph_positive_support": graph_pos.get(predicted, 0.0),
        "pred_graph_contradiction": graph_contra.get(predicted, 0.0),
        "bayes_top1": bayes_top1,
        "bayes_top_log_score": bayes_top_score,
        "bayes_margin": top_margin(bayes_scores),
        "pred_bayes_log_score": pred_bayes_score,
        "pred_bayes_rank": rank_of(bayes_scores, predicted),
        "pred_bayes_posterior": bayes_post.get(predicted, 0.0),
    })

ledger_scores = case_run_outcomes.apply(enrich_with_ledger_scores, axis=1)
case_run_outcomes = pd.concat([case_run_outcomes, ledger_scores], axis=1)

# Label-free suspicion signals. These are computed from a single terminal trajectory only.
case_run_outcomes["signal_llm_mlp_disagree"] = ~case_run_outcomes["llm_mlp_agreement"].astype(bool)
case_run_outcomes["signal_cap_hit"] = case_run_outcomes["stop_reason"].eq("max_requests_reached")
case_run_outcomes["signal_uncertain_mlp"] = (case_run_outcomes["final_mlp_entropy"] > 0.10) | (case_run_outcomes["final_mlp_margin"] < 0.20)
case_run_outcomes["signal_early_uncertain_stop"] = (case_run_outcomes["num_requests"] <= 3) & ((case_run_outcomes["final_mlp_entropy"] > 0.10) | (case_run_outcomes["final_mlp_margin"] < 0.70))
case_run_outcomes["signal_graph_conflict"] = (case_run_outcomes["pred_graph_rank"] > 5) | (case_run_outcomes["pred_graph_score"] < 0.0)
case_run_outcomes["signal_bayes_conflict"] = (case_run_outcomes["pred_bayes_rank"] > 5) | (case_run_outcomes["pred_bayes_posterior"] < 0.05)
case_run_outcomes["signal_ledger_disagrees"] = (case_run_outcomes["graph_top1"].ne(case_run_outcomes["predicted_pathology"])) & (case_run_outcomes["bayes_top1"].ne(case_run_outcomes["predicted_pathology"]))
SIGNAL_COLS = [
    "signal_llm_mlp_disagree",
    "signal_cap_hit",
    "signal_uncertain_mlp",
    "signal_early_uncertain_stop",
    "signal_graph_conflict",
    "signal_bayes_conflict",
    "signal_ledger_disagrees",
]
case_run_outcomes["suspicion_signal_count"] = case_run_outcomes[SIGNAL_COLS].sum(axis=1)

case_run_outcomes.head()


## 4. Trajectory Divergence Measurement

In [ ]:
def turn_records_for_run(run_id, traces, pred_df):
    pred_by_case = pred_df.set_index("case_id")
    records = []
    for cid, trace_obj in traces.items():
        if cid not in common_cases:
            continue
        final_row = pred_by_case.loc[cid]
        evidence = {}
        trace = trace_obj.get("trace", [])
        if trace:
            evidence.update(parse_initial_ledger(trace[0].get("visible_context_before", {}).get("visible_ledger", "")))
        request_count_before = 0
        for turn in trace:
            agent = turn.get("agent_response", {}) or {}
            stop_signal = turn.get("stop_signal", {}) or {}
            mlp = turn.get("mlp_feedback", {}) or {}
            det = turn.get("deterministic_state", {}) or {}
            shortlist = turn.get("shortlist", []) or []
            decision = agent.get("decision") or "unknown"
            requested = agent.get("requested_evidence_id")
            action_key = "STOP" if decision == "stop" else (requested or decision)
            shortlist_roots = [x.get("root_evidence_id") for x in shortlist if isinstance(x, dict)]
            requested_rank = (shortlist_roots.index(requested) + 1) if requested in shortlist_roots else np.nan
            records.append({
                "run_id": run_id,
                "case_id": cid,
                "turn_index": turn.get("turn_index"),
                "prefix_signature": evidence_signature(evidence),
                "requests_before": request_count_before,
                "action_type": decision,
                "action_key": action_key,
                "requested_root": requested,
                "requested_rank_in_shortlist": requested_rank,
                "agent_predicted_pathology": agent.get("predicted_pathology"),
                "agent_confidence": safe_float(agent.get("confidence")),
                "agent_ranked_differential": json.dumps(agent.get("ranked_differential", [])),
                "mlp_top1": mlp.get("top1"),
                "mlp_confidence": safe_float(mlp.get("confidence")),
                "mlp_margin": safe_float(mlp.get("margin")),
                "mlp_entropy": safe_float(mlp.get("entropy")),
                "mlp_stability_turns": safe_float(mlp.get("stability_turns")),
                "deterministic_top1": (det.get("top_candidates") or [[None]])[0][0] if det.get("top_candidates") else None,
                "deterministic_margin": safe_float(det.get("margin")),
                "deterministic_unresolved_mass": safe_float(det.get("unresolved_mass")),
                "top_shortlist_score": safe_float(stop_signal.get("top_shortlist_score")),
                "selected_stop_rule_fired": as_bool(stop_signal.get("selected_stop_rule_fired")),
                "stop_signal_level": stop_signal.get("level"),
                "final_prediction": trace_obj.get("predicted_pathology"),
                "true_pathology": trace_obj.get("true_pathology"),
                "final_correct": bool(trace_obj.get("predicted_pathology") == trace_obj.get("true_pathology")),
                "final_num_requests": trace_obj.get("num_requests"),
            })
            payload = turn.get("reveal_payload")
            if isinstance(payload, dict):
                root = payload.get("root_evidence_id")
                status = normalize_status(payload.get("status"))
                if root and status in {"present", "absent"}:
                    evidence[root] = status
                    request_count_before += 1
    return records

turn_records = []
for run_id in RUN_ORDER:
    turn_records.extend(turn_records_for_run(run_id, run_trace_maps[run_id], run_prediction_frames[run_id]))
turn_level = pd.DataFrame(turn_records)
print(turn_level.shape)
turn_level.head()


In [ ]:
state_rows = []
for (cid, prefix), group in turn_level.groupby(["case_id", "prefix_signature"]):
    if len(group) < 2:
        continue
    action_counts = group["action_key"].value_counts().to_dict()
    action_types = group["action_type"].value_counts().to_dict()
    unique_actions = len(action_counts)
    if unique_actions <= 1:
        continue
    has_stop = any(a == "STOP" for a in action_counts)
    has_request = any(a != "STOP" for a in action_counts)
    if has_stop and has_request:
        divergence_type = "stop_vs_request"
    elif has_request:
        divergence_type = "request_choice"
    else:
        divergence_type = "other"
    state_rows.append({
        "case_id": cid,
        "prefix_signature": prefix,
        "runs_at_state": len(group),
        "requests_before": int(group["requests_before"].min()),
        "unique_actions": unique_actions,
        "action_counts": json.dumps(action_counts, sort_keys=True),
        "action_type_counts": json.dumps(action_types, sort_keys=True),
        "divergence_type": divergence_type,
        "downstream_predictions": json.dumps(group.groupby("run_id")["final_prediction"].first().to_dict(), sort_keys=True),
        "downstream_correct_by_run": json.dumps(group.groupby("run_id")["final_correct"].first().to_dict(), sort_keys=True),
        "downstream_correct_instability": group.groupby("run_id")["final_correct"].first().nunique() > 1,
        "mean_mlp_confidence": group["mlp_confidence"].mean(),
        "mean_mlp_margin": group["mlp_margin"].mean(),
        "mean_mlp_entropy": group["mlp_entropy"].mean(),
        "mean_deterministic_margin": group["deterministic_margin"].mean(),
        "mean_unresolved_mass": group["deterministic_unresolved_mass"].mean(),
        "mean_top_shortlist_score": group["top_shortlist_score"].mean(),
    })
state_divergence = pd.DataFrame(state_rows).sort_values(["case_id", "requests_before", "divergence_type"])
print(state_divergence["divergence_type"].value_counts(dropna=False).to_string())
state_divergence.head(10)


In [ ]:
def action_sequence_for_trace(trace_obj):
    seq = []
    requests = []
    outcomes = []
    for turn in trace_obj.get("trace", []):
        agent = turn.get("agent_response", {}) or {}
        decision = agent.get("decision") or "unknown"
        if decision == "stop":
            seq.append("STOP")
        elif decision == "request":
            root = agent.get("requested_evidence_id") or "REQUEST_UNKNOWN"
            seq.append(root)
            payload = turn.get("reveal_payload")
            if isinstance(payload, dict):
                status = normalize_status(payload.get("status"))
                requests.append(root)
                outcomes.append(f"{root}:{status}")
        else:
            seq.append(decision)
    if not seq or seq[-1] != "STOP":
        seq.append("AUTO_STOP")
    return seq, requests, outcomes

case_rows = []
for cid in sorted(common_cases):
    pred_by_run = {}
    correct_by_run = {}
    request_count_by_run = {}
    seq_by_run = {}
    request_seq_by_run = {}
    outcome_seq_by_run = {}
    true_pathology = None
    for run_id in RUN_ORDER:
        trace_obj = run_trace_maps[run_id][cid]
        seq, roots, outcomes = action_sequence_for_trace(trace_obj)
        seq_by_run[run_id] = seq
        request_seq_by_run[run_id] = roots
        outcome_seq_by_run[run_id] = outcomes
        pred_by_run[run_id] = trace_obj.get("predicted_pathology")
        true_pathology = trace_obj.get("true_pathology")
        correct_by_run[run_id] = pred_by_run[run_id] == true_pathology
        request_count_by_run[run_id] = trace_obj.get("num_requests")
    max_len = max(len(seq) for seq in seq_by_run.values())
    first_divergence_turn = None
    first_divergence_tokens = None
    for idx in range(max_len):
        tokens = {run_id: (seq[idx] if idx < len(seq) else "AUTO_STOP") for run_id, seq in seq_by_run.items()}
        if len(set(tokens.values())) > 1:
            first_divergence_turn = idx + 1
            first_divergence_tokens = tokens
            break
    if first_divergence_turn is None:
        divergence_type = "none"
    else:
        vals = set(first_divergence_tokens.values())
        if any(v in {"STOP", "AUTO_STOP"} for v in vals) and any(v not in {"STOP", "AUTO_STOP"} for v in vals):
            divergence_type = "stop_vs_request"
        elif any(v.startswith("E_") for v in vals):
            divergence_type = "request_choice"
        else:
            divergence_type = "other"
    vote = Counter(pred_by_run.values()).most_common()
    majority_pred = vote[0][0]
    case_rows.append({
        "case_id": cid,
        "true_pathology": true_pathology,
        "unique_predictions": len(set(pred_by_run.values())),
        "prediction_by_run": json.dumps(pred_by_run, sort_keys=True),
        "correct_by_run": json.dumps(correct_by_run, sort_keys=True),
        "requests_by_run": json.dumps(request_count_by_run, sort_keys=True),
        "min_requests": min(request_count_by_run.values()),
        "max_requests": max(request_count_by_run.values()),
        "request_range": max(request_count_by_run.values()) - min(request_count_by_run.values()),
        "first_divergence_turn": first_divergence_turn,
        "first_divergence_type": divergence_type,
        "first_divergence_actions": json.dumps(first_divergence_tokens, sort_keys=True) if first_divergence_tokens else "{}",
        "all_runs_same_prediction": len(set(pred_by_run.values())) == 1,
        "all_runs_same_correctness": len(set(correct_by_run.values())) == 1,
        "any_run_correct": any(correct_by_run.values()),
        "all_runs_correct": all(correct_by_run.values()),
        "num_correct_runs": int(sum(correct_by_run.values())),
        "majority_prediction": majority_pred,
        "majority_correct": majority_pred == true_pathology,
        "n13_correct": correct_by_run.get("n13_frozen"),
        "n24_correct": correct_by_run.get("n24_base"),
        "n25_any_correct": any(correct_by_run[r] for r in ["n25_r01", "n25_r02", "n25_r03"]),
        "n13_miss_any_other_correct": (not correct_by_run.get("n13_frozen")) and any(v for k, v in correct_by_run.items() if k != "n13_frozen"),
    })
case_divergence = pd.DataFrame(case_rows)
case_divergence.head()


## 5. Branch Policy Simulation

In [ ]:
run_level_summary = []
for run_id, group in case_run_outcomes.groupby("run_id"):
    run_level_summary.append({
        "run_id": run_id,
        "accuracy": group["correct"].mean(),
        "num_correct": int(group["correct"].sum()),
        "top3_accuracy": group["top3_correct"].mean(),
        "top5_accuracy": group["top5_correct"].mean(),
        "mean_requests": group["num_requests"].mean(),
        "median_requests": group["num_requests"].median(),
        "llm_mlp_agreement_rate": group["llm_mlp_agreement"].mean(),
        "mean_suspicion_signal_count": group["suspicion_signal_count"].mean(),
        "stop_distribution": json.dumps(group["stop_reason"].value_counts().to_dict(), sort_keys=True),
    })
run_level_summary = pd.DataFrame(run_level_summary).set_index("run_id").loc[RUN_ORDER].reset_index()
run_level_summary


In [ ]:
def trigger_fires(base_row, trigger_name):
    count = int(base_row["suspicion_signal_count"])
    early = bool(base_row["signal_early_uncertain_stop"])
    graph_bayes = bool(base_row["signal_graph_conflict"] and base_row["signal_bayes_conflict"])
    if trigger_name == "never":
        return False
    if trigger_name == "disagreement_or_cap":
        return bool(base_row["signal_llm_mlp_disagree"] or base_row["signal_cap_hit"])
    if trigger_name == "premature_stop_guard":
        return bool(early or (base_row["stop_reason"] == "agent_stop" and base_row["final_mlp_confidence"] < 0.95 and base_row["final_mlp_margin"] < 0.85))
    if trigger_name == "graph_bayes_conflict":
        return graph_bayes
    if trigger_name == "strict_final_conflict":
        return count >= 3
    if trigger_name == "hybrid_suspicion_v1":
        return bool(count >= 2 or (early and graph_bayes))
    if trigger_name == "broad_suspicion":
        return count >= 1
    raise ValueError(trigger_name)


def minmax_list(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return []
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-12:
        return [0.0 for _ in values]
    return [float((v - lo) / (hi - lo)) for v in arr]


CASE_RUN_RECORDS = {}
for row in case_run_outcomes.to_dict("records"):
    CASE_RUN_RECORDS[(row["case_id"], row["run_id"])] = row


def candidate_pool_for(case_id, base_run, branch_budget):
    base = dict(CASE_RUN_RECORDS[(case_id, base_run)])
    base["candidate_role"] = "base"
    rows = [base]
    alternatives = [r for r in RUN_ORDER if r != base_run]
    for alt in alternatives[:branch_budget]:
        item = dict(CASE_RUN_RECORDS[(case_id, alt)])
        item["candidate_role"] = "branch"
        rows.append(item)
    return rows


def score_candidate_pool(pool):
    pool = [dict(row) for row in pool]
    vote_counts = Counter(row["predicted_pathology"] for row in pool)
    graph_scaled = minmax_list([row["pred_graph_score"] for row in pool])
    bayes_scaled = minmax_list([row["pred_bayes_log_score"] for row in pool])
    mlp_conf_scaled = minmax_list([row["final_mlp_confidence"] for row in pool])
    mlp_margin_scaled = minmax_list([row["final_mlp_margin"] for row in pool])
    entropy_low_scaled = minmax_list([-row["final_mlp_entropy"] for row in pool])
    request_low_scaled = minmax_list([-row["num_requests"] for row in pool])
    for i, row in enumerate(pool):
        row["vote_share"] = vote_counts[row["predicted_pathology"]] / len(pool)
        row["graph_scaled"] = graph_scaled[i]
        row["bayes_scaled"] = bayes_scaled[i]
        row["mlp_conf_scaled"] = mlp_conf_scaled[i]
        row["mlp_margin_scaled"] = mlp_margin_scaled[i]
        row["entropy_low_scaled"] = entropy_low_scaled[i]
        row["request_low_scaled"] = request_low_scaled[i]
        row["agreement_bonus"] = float(bool(row["llm_mlp_agreement"]))
        row["fused_branch_score"] = (
            1.30 * row["vote_share"]
            + 0.80 * row["graph_scaled"]
            + 0.80 * row["bayes_scaled"]
            + 0.45 * row["mlp_conf_scaled"]
            + 0.35 * row["mlp_margin_scaled"]
            + 0.25 * row["entropy_low_scaled"]
            + 0.15 * row["agreement_bonus"]
            + 0.10 * row["request_low_scaled"]
        )
    return pool


def choose_candidate(pool, judge_name, base_run):
    scored = score_candidate_pool(pool)
    run_rank = {run_id: idx for idx, run_id in enumerate(RUN_ORDER)}
    base = next(row for row in scored if row["run_id"] == base_run)
    if judge_name == "base_reference":
        chosen = base
    elif judge_name == "majority_vote":
        counts = Counter(row["predicted_pathology"] for row in scored)
        top_count = max(counts.values())
        top_preds = {pred for pred, count in counts.items() if count == top_count}
        if base["predicted_pathology"] in top_preds:
            chosen = base
        else:
            chosen = sorted([row for row in scored if row["predicted_pathology"] in top_preds], key=lambda r: run_rank[r["run_id"]])[0]
    elif judge_name == "highest_mlp_confidence":
        chosen = sorted(scored, key=lambda r: (-r["final_mlp_confidence"], run_rank[r["run_id"]]))[0]
    elif judge_name == "highest_mlp_margin":
        chosen = sorted(scored, key=lambda r: (-r["final_mlp_margin"], run_rank[r["run_id"]]))[0]
    elif judge_name == "highest_graph_support":
        chosen = sorted(scored, key=lambda r: (-r["pred_graph_score"], run_rank[r["run_id"]]))[0]
    elif judge_name == "highest_bayes_posterior":
        chosen = sorted(scored, key=lambda r: (-r["pred_bayes_posterior"], run_rank[r["run_id"]]))[0]
    elif judge_name == "fused_consensus_judge":
        chosen = sorted(scored, key=lambda r: (-r["fused_branch_score"], run_rank[r["run_id"]]))[0]
    elif judge_name == "cautious_fused_judge":
        best = sorted(scored, key=lambda r: (-r["fused_branch_score"], run_rank[r["run_id"]]))[0]
        chosen = best if float(best["fused_branch_score"]) >= float(base["fused_branch_score"]) + 0.20 else base
    else:
        raise ValueError(judge_name)
    return chosen, scored

TRIGGERS = ["never", "disagreement_or_cap", "premature_stop_guard", "graph_bayes_conflict", "strict_final_conflict", "hybrid_suspicion_v1", "broad_suspicion"]
JUDGES = ["base_reference", "majority_vote", "highest_mlp_confidence", "highest_mlp_margin", "highest_graph_support", "highest_bayes_posterior", "fused_consensus_judge", "cautious_fused_judge"]
BRANCH_BUDGETS = [0, 1, 2, 4]

policy_case_rows = []
candidate_score_rows = []
case_ids = sorted(common_cases)
for base_run in RUN_ORDER:
    for trigger_name in TRIGGERS:
        for branch_budget in BRANCH_BUDGETS:
            for judge_name in JUDGES:
                if branch_budget == 0 and (trigger_name != "never" or judge_name != "base_reference"):
                    continue
                for cid in case_ids:
                    base_row = candidate_pool_for(cid, base_run, 0)[0]
                    fired = trigger_fires(base_row, trigger_name) if branch_budget > 0 else False
                    effective_budget = branch_budget if fired else 0
                    pool = candidate_pool_for(cid, base_run, effective_budget)
                    chosen, scored_pool = choose_candidate(pool, judge_name if effective_budget > 0 else "base_reference", base_run)
                    selected_pred = chosen["predicted_pathology"]
                    correct = selected_pred == base_row["true_pathology"]
                    total_branch_requests = int(base_row["num_requests"])
                    if fired:
                        total_branch_requests += int(sum(row["num_requests"] for row in scored_pool if row["candidate_role"] == "branch"))
                    policy_name = f"trigger={trigger_name}|budget={branch_budget}|judge={judge_name}"
                    policy_case_rows.append({
                        "policy_name": policy_name,
                        "base_run": base_run,
                        "trigger_name": trigger_name,
                        "branch_budget": branch_budget,
                        "judge_name": judge_name,
                        "case_id": cid,
                        "true_pathology": base_row["true_pathology"],
                        "base_prediction": base_row["predicted_pathology"],
                        "base_correct": bool(base_row["correct"]),
                        "trigger_fired": bool(fired),
                        "branches_spawned": int(effective_budget),
                        "selected_run": chosen.get("run_id", base_run),
                        "selected_prediction": selected_pred,
                        "selected_correct": bool(correct),
                        "selected_requests": int(chosen.get("num_requests", base_row["num_requests"])),
                        "total_branch_requests": total_branch_requests,
                        "base_suspicion_signal_count": int(base_row["suspicion_signal_count"]),
                    })
                    if fired and judge_name in {"fused_consensus_judge", "cautious_fused_judge"}:
                        for row in scored_pool:
                            candidate_score_rows.append({
                                "policy_name": policy_name,
                                "base_run": base_run,
                                "run_id": row["run_id"],
                                "case_id": row["case_id"],
                                "true_pathology": row["true_pathology"],
                                "predicted_pathology": row["predicted_pathology"],
                                "correct": bool(row["correct"]),
                                "candidate_role": row["candidate_role"],
                                "vote_share": row["vote_share"],
                                "pred_graph_score": row["pred_graph_score"],
                                "pred_graph_rank": row["pred_graph_rank"],
                                "pred_bayes_posterior": row["pred_bayes_posterior"],
                                "pred_bayes_rank": row["pred_bayes_rank"],
                                "final_mlp_confidence": row["final_mlp_confidence"],
                                "final_mlp_margin": row["final_mlp_margin"],
                                "final_mlp_entropy": row["final_mlp_entropy"],
                                "llm_mlp_agreement": bool(row["llm_mlp_agreement"]),
                                "num_requests": row["num_requests"],
                                "fused_branch_score": row["fused_branch_score"],
                            })

policy_case_results = pd.DataFrame(policy_case_rows)
candidate_branch_scores = pd.DataFrame(candidate_score_rows)
print(policy_case_results.shape, candidate_branch_scores.shape)
policy_case_results.head()


In [ ]:
def summarize_policy_cases(df, scope_name):
    return {
        "scope": scope_name,
        "num_decisions": len(df),
        "num_correct": int(df["selected_correct"].sum()),
        "accuracy": df["selected_correct"].mean(),
        "base_accuracy": df["base_correct"].mean(),
        "wins_vs_base": int(((~df["base_correct"]) & df["selected_correct"]).sum()),
        "regressions_vs_base": int((df["base_correct"] & (~df["selected_correct"])).sum()),
        "branch_rate": df["trigger_fired"].mean(),
        "mean_branches_spawned": df["branches_spawned"].mean(),
        "mean_selected_requests": df["selected_requests"].mean(),
        "mean_total_branch_requests": df["total_branch_requests"].mean(),
    }

summary_rows = []
for policy_name, group in policy_case_results.groupby("policy_name"):
    row = summarize_policy_cases(group, "all_base_runs")
    row["policy_name"] = policy_name
    summary_rows.append(row)
    for base_run, bgroup in group.groupby("base_run"):
        brow = summarize_policy_cases(bgroup, base_run)
        brow["policy_name"] = policy_name
        summary_rows.append(brow)
branch_policy_summary = pd.DataFrame(summary_rows)
branch_policy_summary = branch_policy_summary[["policy_name", "scope", "num_decisions", "num_correct", "accuracy", "base_accuracy", "wins_vs_base", "regressions_vs_base", "branch_rate", "mean_branches_spawned", "mean_selected_requests", "mean_total_branch_requests"]]
branch_policy_summary.sort_values(["scope", "accuracy", "branch_rate"], ascending=[True, False, True]).head(20)


## 6. Upper Bounds And Candidate Selection

In [ ]:
oracle_rows = []
for base_run in RUN_ORDER:
    for branch_budget in BRANCH_BUDGETS:
        for trigger_name in TRIGGERS:
            if branch_budget == 0 and trigger_name != "never":
                continue
            rows = []
            for cid in sorted(common_cases):
                base_row = candidate_pool_for(cid, base_run, 0)[0]
                fired = trigger_fires(base_row, trigger_name) if branch_budget > 0 else False
                effective_budget = branch_budget if fired else 0
                pool = candidate_pool_for(cid, base_run, effective_budget)
                oracle_correct = any(bool(row["correct"]) for row in pool)
                rows.append({"oracle_correct": oracle_correct, "base_correct": bool(base_row["correct"]), "trigger_fired": fired, "branches_spawned": effective_budget})
            temp = pd.DataFrame(rows)
            oracle_rows.append({
                "base_run": base_run,
                "trigger_name": trigger_name,
                "branch_budget": branch_budget,
                "oracle_accuracy": temp["oracle_correct"].mean(),
                "base_accuracy": temp["base_correct"].mean(),
                "oracle_wins_vs_base": int(((~temp["base_correct"]) & temp["oracle_correct"]).sum()),
                "oracle_missed_even_with_pool": int((~temp["oracle_correct"]).sum()),
                "branch_rate": temp["trigger_fired"].mean(),
                "mean_branches_spawned": temp["branches_spawned"].mean(),
            })
branch_oracle_summary = pd.DataFrame(oracle_rows)
branch_oracle_summary.sort_values(["oracle_accuracy", "branch_rate"], ascending=[False, True]).head(20)


In [ ]:
# Pre-registered next-live candidate: conservative trigger, one alternate branch, cautious fused judge.
# It is not selected because it wins on the 49-case labels; it encodes the intended behavior:
# spawn sparingly, then require a fused graph/Bayes/MLP advantage before overriding the base branch.
PRE_REGISTERED_POLICY_NAME = "trigger=hybrid_suspicion_v1|budget=1|judge=cautious_fused_judge"

all_scope = branch_policy_summary[branch_policy_summary["scope"].eq("all_base_runs")].copy()
best_observed = all_scope.sort_values(["accuracy", "branch_rate", "mean_total_branch_requests"], ascending=[False, True, True]).iloc[0].to_dict()
pre_registered = all_scope[all_scope["policy_name"].eq(PRE_REGISTERED_POLICY_NAME)].iloc[0].to_dict()

# Conservative Pareto view among label-free variants. This still uses labels for evaluation, so it is diagnostic.
pareto = all_scope.copy().sort_values(["branch_rate", "mean_total_branch_requests"])
pareto_rows = []
best_acc_so_far = -1.0
for _, row in pareto.iterrows():
    if row["accuracy"] > best_acc_so_far + 1e-12:
        pareto_rows.append(row)
        best_acc_so_far = row["accuracy"]
branch_policy_pareto = pd.DataFrame(pareto_rows)

selection_record = {
    "notebook": "26_offline_branching_trajectory_lab.ipynb",
    "artifact_root": str(ARTIFACT_ROOT),
    "inputs_used": {run_id: str(path) for run_id, path in RUN_SPECS.items()},
    "graph_edge_path": str(GRAPH_EDGE_PATH),
    "bayes_likelihood_path": str(BAYES_LIKELIHOOD_PATH),
    "run_order": RUN_ORDER,
    "pre_registered_next_live_candidate": pre_registered,
    "best_observed_diagnostic_policy": best_observed,
    "selection_status": "diagnostic_only_not_promoted",
    "reason": "Observed branch trajectories are reused as offline alternatives; labels are used only for analysis of ceilings and policy curves. A live confirmation notebook is required before promotion.",
}
print(json.dumps(selection_record, indent=2)[:4000])
branch_policy_pareto.head(20)


## 7. Figures

In [ ]:
if plt is not None:
    plt.style.use("default")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(run_level_summary["run_id"], run_level_summary["accuracy"], color=["#31588a", "#4f8f6f", "#8a6fb0", "#b05a5a", "#b08a48"])
    ax.set_ylim(0.75, 1.0)
    ax.set_ylabel("Accuracy")
    ax.set_title("Observed base trajectory accuracy")
    for idx, row in run_level_summary.iterrows():
        ax.text(idx, row["accuracy"] + 0.006, f"{int(row['num_correct'])}/49", ha="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "observed_run_accuracy.png", dpi=180)
    plt.close(fig)

    div_counts = case_divergence["first_divergence_type"].value_counts().reindex(["none", "request_choice", "stop_vs_request", "other"]).fillna(0)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(div_counts.index, div_counts.values, color="#5d7f99")
    ax.set_ylabel("Cases")
    ax.set_title("First trajectory divergence type across five runs")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "first_divergence_type_counts.png", dpi=180)
    plt.close(fig)

    plot_df = all_scope[~all_scope["policy_name"].str.contains("budget=0")].copy()
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(plot_df["branch_rate"], plot_df["accuracy"], c=plot_df["mean_total_branch_requests"], cmap="viridis", s=45, alpha=0.85)
    ax.scatter([pre_registered["branch_rate"]], [pre_registered["accuracy"]], marker="*", s=180, color="#d94801", label="pre-registered candidate")
    ax.set_xlabel("Branch trigger rate")
    ax.set_ylabel("Pooled accuracy across base runs")
    ax.set_title("Branch policy frontier, diagnostic offline simulation")
    ax.legend(loc="lower right")
    cb = fig.colorbar(sc, ax=ax)
    cb.set_label("Mean total branch requests")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "branch_policy_frontier.png", dpi=180)
    plt.close(fig)

    heat = case_run_outcomes.pivot(index="case_id", columns="run_id", values="correct").loc[:, RUN_ORDER]
    heat = heat.astype(int)
    fig, ax = plt.subplots(figsize=(7, 10))
    ax.imshow(heat.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(RUN_ORDER)))
    ax.set_xticklabels(RUN_ORDER, rotation=35, ha="right")
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=6)
    ax.set_title("Correctness matrix across observed trajectories")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "trajectory_correctness_matrix.png", dpi=180)
    plt.close(fig)

    print(f"figures written to {FIGURES_DIR}")
else:
    print("matplotlib not available; skipped figures")


## 8. Final Summary And Artifact Contract

In [ ]:
run_level_summary.to_csv(ARTIFACT_ROOT / "run_level_summary.csv", index=False)
case_run_outcomes.to_csv(ARTIFACT_ROOT / "case_run_outcomes.csv", index=False)
turn_level.to_csv(ARTIFACT_ROOT / "turn_level_branch_features.csv", index=False)
state_divergence.to_csv(ARTIFACT_ROOT / "state_divergence_points.csv", index=False)
case_divergence.to_csv(ARTIFACT_ROOT / "case_divergence_summary.csv", index=False)
branch_policy_summary.to_csv(ARTIFACT_ROOT / "branch_policy_summary.csv", index=False)
branch_policy_pareto.to_csv(ARTIFACT_ROOT / "branch_policy_pareto.csv", index=False)
branch_oracle_summary.to_csv(ARTIFACT_ROOT / "branch_oracle_summary.csv", index=False)
policy_case_results.to_csv(ARTIFACT_ROOT / "branch_policy_case_results.csv", index=False)
candidate_branch_scores.to_csv(ARTIFACT_ROOT / "candidate_branch_scores.csv", index=False)

hard_cases = case_divergence[(~case_divergence["all_runs_correct"]) | (~case_divergence["all_runs_same_prediction"])].copy()
hard_audits = []
for _, row in hard_cases.iterrows():
    cid = row["case_id"]
    run_rows = case_run_outcomes[case_run_outcomes["case_id"].eq(cid)].set_index("run_id").loc[RUN_ORDER]
    hard_audits.append({
        "case_id": cid,
        "true_pathology": row["true_pathology"],
        "first_divergence_turn": None if pd.isna(row["first_divergence_turn"]) else int(row["first_divergence_turn"]),
        "first_divergence_type": row["first_divergence_type"],
        "first_divergence_actions": parse_jsonish(row["first_divergence_actions"], default={}),
        "prediction_by_run": parse_jsonish(row["prediction_by_run"], default={}),
        "correct_by_run": parse_jsonish(row["correct_by_run"], default={}),
        "requests_by_run": parse_jsonish(row["requests_by_run"], default={}),
        "terminal_branch_features": run_rows[[
            "predicted_pathology", "correct", "num_requests", "stop_reason", "final_mlp_confidence", "final_mlp_margin", "final_mlp_entropy",
            "llm_mlp_agreement", "pred_graph_rank", "pred_graph_score", "pred_bayes_rank", "pred_bayes_posterior", "suspicion_signal_count"
        ]].reset_index().to_dict("records"),
    })
with (ARTIFACT_ROOT / "hard_case_branch_audits.json").open("w") as f:
    json.dump(hard_audits, f, indent=2)

with (ARTIFACT_ROOT / "recommended_branching_policy.json").open("w") as f:
    json.dump(selection_record, f, indent=2)

config = {
    "notebook": "26_offline_branching_trajectory_lab.ipynb",
    "created_for": "offline multi-agent branching trajectory analysis",
    "artifact_root": str(ARTIFACT_ROOT),
    "run_order": RUN_ORDER,
    "run_specs": {run_id: str(path) for run_id, path in RUN_SPECS.items()},
    "graph_clip": GRAPH_CLIP,
    "triggers": TRIGGERS,
    "judges": JUDGES,
    "branch_budgets": BRANCH_BUDGETS,
    "pre_registered_next_live_candidate": PRE_REGISTERED_POLICY_NAME,
    "no_live_api_calls": True,
}
with (ARTIFACT_ROOT / "resolved_run_config.json").open("w") as f:
    json.dump(config, f, indent=2)

artifact_contract = [
    "resolved_run_config.json",
    "run_level_summary.csv",
    "case_run_outcomes.csv",
    "turn_level_branch_features.csv",
    "state_divergence_points.csv",
    "case_divergence_summary.csv",
    "branch_policy_summary.csv",
    "branch_policy_pareto.csv",
    "branch_oracle_summary.csv",
    "branch_policy_case_results.csv",
    "candidate_branch_scores.csv",
    "hard_case_branch_audits.json",
    "recommended_branching_policy.json",
]
missing_outputs = [name for name in artifact_contract if not (ARTIFACT_ROOT / name).exists()]
if missing_outputs:
    raise AssertionError(f"Missing outputs: {missing_outputs}")

summary = {
    "runs": len(RUN_ORDER),
    "cases": len(common_cases),
    "run_accuracy": run_level_summary.set_index("run_id")["accuracy"].to_dict(),
    "case_prediction_instability": int((~case_divergence["all_runs_same_prediction"]).sum()),
    "case_correctness_instability": int((~case_divergence["all_runs_same_correctness"]).sum()),
    "oracle_best_of_5_accuracy": float(case_divergence["any_run_correct"].mean()),
    "majority_vote_accuracy": float(case_divergence["majority_correct"].mean()),
    "n13_miss_any_other_correct": int(case_divergence["n13_miss_any_other_correct"].sum()),
    "state_divergence_points": int(len(state_divergence)),
    "impactful_state_divergence_points": int(state_divergence["downstream_correct_instability"].sum()) if len(state_divergence) else 0,
    "pre_registered_policy": pre_registered,
    "best_observed_diagnostic_policy": best_observed,
}
with (ARTIFACT_ROOT / "analysis_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2)[:5000])
